# out of date (aug 5 2025):

This demo does not represent the latest implementation.  Language could be tuned up a bit, needs some more thinking about the intended audience. 

# HGLM Mandrill Example

This jupyter notebook runs through a toy example of a Monkey's face.  Its a great place to get started for technical and non-technical readers alike.  By the end of this notebook you'll have a sense of what problem we're solving and how we approach it.  The included code is helpful if you want to repeat or tweak the experiment, but its not at all necessary to get started.  

# Generating a toy set of images

We begin by loading a single image and coping it `num_img` times to simulate an image population.  We add a bit of noise to each image to simulate having images of different monkey faces.  Notice that the images here are in a registered space, each pixel corresponds to the same anatomical location.  In other words, if we were to imagine stacking the images on top of each other the left eye of the monkey, and every other part, would align across images.

In [ ]:
import glow
import pathlib
import numpy as np
from hglm_image_demo import imshow_rgb
import matplotlib.pyplot as plt

seed = 0
num_img = 5
noise_scale = 1

folder = pathlib.Path(glow.__file__).parents[1] / 'test' / 'data'

# simple explanatory feature, easier graphing
x = np.arange(num_img).reshape((1, num_img))
contrast = np.array([True])

exp = glow.experiment.Experiment.from_search(folder=folder,
                                             sbj_regex='mandrill_small',
                                             img_glob_dict={'rgb': 'mandrill_small.png'},
                                             x=x, contrast=contrast, add_bias=True)
exp.bootstrap_img(n=num_img, seed=seed, noise_scale=noise_scale)

In [ ]:
fig, ax = plt.subplots(1, num_img)
fig.set_size_inches(14, 5)
for idx in range(num_img):
    plt.sca(ax[idx])
    imshow_rgb(exp.y[:, idx, :], exp.mask_idx, label=f'img{idx} (x={idx})')

# Problem Statement (What is an "effect"?)

Suppose you hopped into an MRI each day to record an image of your brain.  You might want to see how your brain is changing as you age.  Maybe its the case that there is some region within the images which grow "lighter" or "darker" with age.  Or, more realistically, you might want to compare images of people with and without an illness.  Maybe its the case that the illness impacts some region of the image consistently across individuals.  

In both cases, we are given a population of images (our $y$ feature, image intensity) as well as some explanatory feature(s) ($x$ feature).   Our goal is to identify some set of contiguous pixels in the image whose $y$ is not-just-coincidnetally-correlated with $x$.  Some effects which might be discovered, per our examples above:
- in the longitudinal example:
    1. in a particular 100 voxels towards the top of my brain
    1. every year I get older then these voxels get "darker"
- in the group comparison example:
    1. there is a small region of 57 voxels
    2. which are consistently "lighter" in people with a given illness

More generally, notice that an effect is characterized by two properties:
1. the spatial **extent** of the effect: where in the image does the effect occur?
1. the **mapping** from explanatory features $x$ to $y$: how does the effect change the images?

Our effect definition is consistent with the General Linear Model [wikipedia](https://en.wikipedia.org/wiki/General_linear_model) applied across different pixels in a population of registered images.

### Problem Statement

Identify all the effects present in a given population of images.  Our ideal output is a set of effects (pairs of extents & mappings) with:
- high sensitivity: we don't miss any voxels contained in an effect
- high specificity: we don't include any non-effect voxels in an effect's extent
- effect regions are whole: an effect's extent isn't split into two distinct output effects
- effect regions are distinct: no two effect extents are aggregated into a single effect

# Imposing an effect


To ask this question properly, one needs a population of images $y$ as well as explanatory features $x$.  We have our monkey images $y$ above, and choose a convenient, arbitrary explanatory feature: consecutive integers ($x=0, x=1, x=2, ...$).  Because we have chosen $x$ arbitrarily there are currently no effects in the data (any correlation between $x$ and $y$ is purely coincidiental).  In this section we modify the images to impose an effect.

### Sampling a spatial extent

Realisitic effects occur in regions with a consistent color.  For example, an illness might target a particular tissue such that its impacted area follows the anatomical contours of the targeted tissue.  To simulate a realistic effect, we choose an extent whose color is as consistent as possible.  This is achieved by sampling an arbitrary pixel in the image and iteratively incorporating the neighboring pixels whose color is as similar as possible to those already collected.  The chosen extent is shown below in bright green.

In [ ]:
# what percentage of voxels are contained in effect
extent_percent = .1

# effect severity (smaller values yield more obvious effects)
p_val = .01

num_vox = np.prod(exp.mask_idx.shape)
num_vox_eff = extent_percent * num_vox
extenter = glow.effect.ExtenterMinVar(num_vox_eff)
mask_eff = extenter(y=exp.y, mask_idx=exp.mask_idx, seed=seed, verbose=True)

In [ ]:
imshow_rgb(exp.y.mean(axis=1), exp.mask_idx, label=None, mask=mask_eff)
plt.suptitle('Effect Extent (in green)');

### Modifying colors to impose correlation between $x$ and $y$

In [ ]:
exp_w_effect, effect, rough = exp.impose_effect(seed=seed, mask=mask_eff, p_val=p_val)

In [ ]:
rgb = (effect.beta[1] @ exp.x)
for idx, c in enumerate(['red', 'green', 'blue']):
    plt.plot(exp.x[1, :], rgb[idx, :], c=c, linewidth=2, label=c)
plt.xlabel('explanatory feature (x)')
plt.ylabel('predicted color (y)')
plt.legend()
plt.suptitle('Mapping from $x$ to $y$');

After imposing the effect on the images, the color of the extent is more-than-coincidentally correlated to $x$.  The graph above characterizes the relationship between $x$ and color.  The same relationship is visualized in the sequence of images below (notice that the effect extent changes color according to the mapping given immediately above).

In [ ]:
fig, ax = plt.subplots(1, num_img)
fig.set_size_inches(14, 5)
for idx in range(num_img):
    plt.sca(ax[idx])
    imshow_rgb(exp_w_effect.y[:, idx, :], exp_w_effect.mask_idx, label=f'img{idx} (x={idx})')

(Note to users who increase the severity of the effect by lowering `p_val`: these rgb values are clipped between 0 and 255 when displayed, you may not be able to observe extereme effects for large and small $x$ values.)

# Estimating the effect

The challenge of identifying the effect is that, a priori, we know neither the extent nor the mapping (i.e. the red-green-blue line plot above).  

In [ ]:
n_perm=50
n_perm_adj=10
alpha=.05

analysis = glow.experiment.AnalysisHGLM(exp_w_effect, n_perm=n_perm, n_perm_adj=n_perm_adj, alpha_fwer=alpha, verbose=True, n_jobs=-1)

## HGLM Step 1: Hierarchical Segmentation

HGLM begins its search by constructing regions which are compelling effect extent candidates.  That is, all the region's pixel colors are close to the predicted colors via the best mapping possible (similar to the red-green-blue line plot above).  HGLM constructs regions to minimize the differences between the estimated and observed colors, mean-squared-error, per region region.  

Towards this end, HGLM uses Agglomerative Clustering, beginning with a one-pixel-per-region segmentation of the entire image.  Adjacent regions, chosen to minimize mean-squared-error, are merged into a new region which replaces each of its constituents.  This merging continues until only one region remains (see gif below).  The process is a close cousin of [Ward's Clustering](https://en.wikipedia.org/wiki/Ward%27s_method).

(In case you have trouble getting the local gif to work, here is a [web link to a similar gif](https://i.ibb.co/6Dbyvjv/hglm-image-demo.gif))

In [ ]:
glow.plot.make_gif(file_out='hglm_image_demo.gif',
                   children=analysis.child_dict[0],
                   mask_idx=exp_w_effect.mask_idx,
                   num_vox=num_vox, n_list=50, fps=5)

<img src="hglm_image_demo.gif" alt="GIF" loop="3" style="width: 300px; height: auto;">

# HGLM Step 2: Signifigance Testing

From among all the regions above, we must determine which contain correlations between $x$ and $y$ which might be conicidental and which HGLM will claim contains an effect.  We're fortunate to import a popular permutation testing framework from [Winkler 2014](https://pubmed.ncbi.nlm.nih.gov/24530839/) to get this done.

Two issues:
- f stats

# HGLM Step 3: 

todo:
- motivate problem of intersecting significant regions
- solution: model selection approach: choose the effect which best explains all intersecting regions

In [ ]:
analysis.llr.shape
plt.imshow(np.log10(analysis.llr[:, 0, -1000:]))
plt.gcf().set_size_inches(10, 5)

In [ ]:
linestyle = ['-', '--', '-.', ':']

def plot_effect(eff_dict, exp):
    # plot extents
    ax = plt.subplot(2, 2, 1), plt.subplot(2, 2, 2)
    for _ax, (label, eff) in zip(ax, eff_dict.items()):
        plt.sca(_ax)
        _ax.set_title(f'extent: {label}')
        imshow_rgb(exp.y.mean(axis=1), exp.mask_idx, label=None, mask=eff.mask)

    # plot mappings
    plt.subplot(2, 1, 2)
    for l, (label, eff) in zip(linestyle, eff_dict.items()):
        rgb = (eff.beta[1] @ exp.x)
        for idx, c in enumerate(['red', 'green', 'blue']):
            plt.plot(exp.x[1, :], rgb[idx, :], c=c, linewidth=2, label=f'{label}: {c}', linestyle=l)
    plt.gca().set_title('Mapping')
    plt.legend()
    plt.xlabel('explanatory feature (x)')
    plt.ylabel('predicted color (y)')
    plt.gcf().set_size_inches(10, 8)

In [ ]:
eff_dict={'ground truth': effect, 
          'estimate': analysis.effect_list[0]}
plot_effect(eff_dict, exp)
plt.suptitle('Estimating an effect');